# D3-02 Generating prospective databases with premise

Audience:
- Participants who already completed the Brightway and `ecoinvent` steps from Days 1 and 2.

Prerequisites:
- `paris-lca-course-2026` exists and contains `ecoinvent-3.12-cutoff`.
- This notebook falls back to the course IAM key `tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=` if `IAM_FILES_KEY` is not set.

Learning goals:
- Build the course scenario list for REMIND-EU.
- Run `premise.NewDatabase(...)` on `ecoinvent 3.12 cutoff`.
- Write the generated scenario databases to Brightway.
- Export a datapackage for optional `trails` work.


## Outline

1. Confirm the input database and credentials.
2. Define the REMIND-EU scenarios used in the course.
3. Generate the updated scenario databases.
4. Write Brightway databases and a `trails` datapackage.
5. Check the exported artifacts.


In [ ]:
import os
from pathlib import Path

import bw2data as bd
import pandas as pd
from premise import NewDatabase

project_name = 'paris-lca-course-2026'
source_db = 'ecoinvent-3.12-cutoff'
system_model = 'cutoff'
source_version = '3.12'
export_name = 'paris-trails-datapackage'
default_iam_key = 'tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo='

bd.projects.set_current(project_name)

biosphere_name = next((name for name in bd.databases if 'biosphere' in name), None)
iam_key = os.environ.get('IAM_FILES_KEY', default_iam_key)

print('Project        :', project_name)
print('Source database:', source_db)
print('Biosphere      :', biosphere_name)
print('Using env key  :', bool(os.environ.get('IAM_FILES_KEY')))
print('Active IAM key :', iam_key)


## Step 1 - Define the course scenarios

We compare two policy pathways across three years, always with `remind-eu` as the source of scenario data.


In [ ]:
scenarios = [
    {'model': 'remind-eu', 'pathway': 'SSP2-NPi', 'year': 2025},
    {'model': 'remind-eu', 'pathway': 'SSP2-NPi', 'year': 2035},
    {'model': 'remind-eu', 'pathway': 'SSP2-NPi', 'year': 2050},
    {'model': 'remind-eu', 'pathway': 'SSP2-PkBudg1000', 'year': 2025},
    {'model': 'remind-eu', 'pathway': 'SSP2-PkBudg1000', 'year': 2035},
    {'model': 'remind-eu', 'pathway': 'SSP2-PkBudg1000', 'year': 2050},
]

scenario_table = pd.DataFrame(scenarios)
scenario_table


## Step 2 - Generate the prospective database object

This step can take a while. Keep the notebook attached to the same `lca-course` kernel while it runs.  
The course IAM key is already embedded below, so you only need `IAM_FILES_KEY` if you want to override it.


In [ ]:
if source_db not in bd.databases:
    raise ValueError(f'Missing source database: {source_db}')

ndb = NewDatabase(
    scenarios=scenarios,
    source_db=source_db,
    source_version=source_version,
    key=iam_key.encode(),
    system_model=system_model,
    biosphere_name=biosphere_name,
)

ndb.update()
print('Scenario update complete.')


## Step 3 - Write Brightway databases and a datapackage

We use deterministic database names so that later Brightway and Activity Browser comparisons stay readable.


In [ ]:
def scenario_db_name(scenario):
    pathway = scenario['pathway'].lower().replace(' ', '-').replace('_', '-')
    model = scenario['model'].lower().replace(' ', '-').replace('_', '-')
    return f"paris-{model}-{pathway}-{scenario['year']}"

db_names = [scenario_db_name(s) for s in scenarios]

for name in db_names:
    if name in bd.databases:
        del bd.databases[name]

ndb.write_db_to_brightway(db_names)
ndb.write_datapackage(name=export_name)

pd.DataFrame({'database_name': db_names})


## Step 4 - Check the exported artifacts

The `premise` datapackage is the bridge to optional `trails` work on Day 4.


In [ ]:
export_path = Path.cwd() / 'export' / 'datapackage' / f'{export_name}.zip'

print('Datapackage path:', export_path)
print('Exists          :', export_path.exists())
print('Generated DBs   :', [name for name in db_names if name in bd.databases])


## Exercises

- Add one more scenario row and predict which part of the workflow will need the most extra time.
- Change the output naming function if you prefer shorter database names.
- Note one risk of writing new scenario databases over old ones without deleting them first.


In [ ]:
# Exercise answer scaffold
notes = {
    'extra_scenario_runtime': '...',
    'naming_change': '...',
    'overwrite_risk': '...',
}
notes
